# core

> Read Ramabana history and score the tool routes in it.

In [ ]:
#| default_exp core

In [ ]:
#| export
from dataclasses import dataclass, asdict
from pathlib import Path
import json, sys

from fastcore.script import call_parse

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail
import tempfile

## Read the archive

Ramabana appends one JSON Lines record per finished turn: the `prompt`, the `reply`, a `state`, and
an `activity` list of every tool call with its arguments, result, and whether it worked.

A turn ends `complete`, `failed`, or `abandoned`. Drona curates only the first — a route nobody saw
through is not one to imitate.

In [ ]:
#| export
RAMABANA_HISTORY = Path.home()/'.config/ramabana/agent-history.jsonl'
DONE = ('complete',)

def _row(line):
    try: return json.loads(line) if line.strip() else None
    except json.JSONDecodeError: return None

def report(fn, *args, **kw):
    "Run `fn`, reporting an expected failure as a message rather than a traceback."
    try: return fn(*args, **kw)
    except (ValueError, FileNotFoundError) as e:
        print(e, file=sys.stderr)
        sys.exit(2)

def match_session(
    turns,   # turns from `read_history`
    session, # session id, id prefix, or `latest`
):
    "The turns of exactly one session, preferring an exact id over a prefix."
    ids = [s for t in turns if (s := t.get('session'))]
    if not ids: raise ValueError('no turn carries a session id')
    if session == 'latest': session = ids[-1]
    hits = ([t for t in turns if t.get('session') == session]
            or [t for t in turns if str(t.get('session') or '').startswith(session)])
    if not hits: raise ValueError(f'no session matches {session!r}')
    if len({t.get('session') for t in hits}) > 1: raise ValueError(f'session {session!r} is ambiguous')
    return hits

def read_history(
    path=RAMABANA_HISTORY, # Ramabana `agent-history.jsonl` path
    session=None,          # session id, id prefix, or `latest`; every session when omitted
    states=DONE,           # turn states to keep; `None` keeps every state
):
    "Ramabana turns, in the order they were appended."
    p = Path(path).expanduser()
    if not p.exists(): raise FileNotFoundError(f'no Ramabana history at {p}')
    rows = [t for line in p.read_text().splitlines() if isinstance(t := _row(line), dict)]
    if not rows: raise ValueError(f'{p} records no turns')
    turns = rows if states is None else [t for t in rows if t.get('state', 'complete') in states]
    if not turns:
        raise ValueError(f'no turn in {p.name} is {" or ".join(states)}; --every-state keeps the rest')
    return match_session(turns, session) if session else turns

A prefix is enough to name a session, and `latest` names the newest one that has a complete turn.

In [ ]:
tmp = Path(tempfile.mkdtemp())
archive = tmp/'agent-history.jsonl'
rows = [{'session': 'sess-aaa', 'state': 'complete', 'prompt': 'first'},
        {'session': 'sess-bbb', 'state': 'abandoned', 'prompt': 'stopped part way'},
        {'session': 'sess-bbb', 'state': 'complete', 'prompt': 'second'}]
archive.write_text('\n'.join(json.dumps(r) for r in rows) + '\n')

test_eq([t['prompt'] for t in read_history(archive)], ['first', 'second'])
test_eq([t['prompt'] for t in read_history(archive, 'sess-a')], ['first'])
test_eq([t['prompt'] for t in read_history(archive, 'latest')], ['second'])
test_eq(len(read_history(archive, states=None)), 3)

The failures a command line meets are reported, not raised as a traceback: a missing archive, an
ambiguous prefix, a prefix that matches nothing, and turns with no session id at all.

In [ ]:
test_fail(lambda: read_history(tmp/'absent.jsonl'), contains='no Ramabana history at')
test_fail(lambda: read_history(archive, 'sess-'), contains='ambiguous')
test_fail(lambda: read_history(archive, 'nope'), contains='no session matches')

anon = tmp/'anon.jsonl'
anon.write_text(json.dumps({'prompt': 'no session id'}) + '\n')
test_fail(lambda: read_history(anon, 'latest'), contains='no turn carries a session id')

A half-written final line is skipped rather than fatal, because Ramabana may be mid-append.

In [ ]:
torn = tmp/'torn.jsonl'
torn.write_text(json.dumps({'session': 's', 'state': 'complete', 'prompt': 'kept'}) + '\n{"session": "s"')
test_eq([t['prompt'] for t in read_history(torn)], ['kept'])

scalar = tmp/'scalar.jsonl'
scalar.write_text('123\n' + json.dumps({'session': 's', 'state': 'complete', 'prompt': 'kept'}) + '\n')
test_eq([t['prompt'] for t in read_history(scalar)], ['kept'])

An archive that holds nothing worth curating says so, rather than reporting the absence of a session id. An id that is also the prefix of a longer one still names its own session.

In [ ]:
empty = tmp/'empty.jsonl'
empty.write_text('')
test_fail(lambda: read_history(empty), contains='records no turns')

stopped = tmp/'stopped.jsonl'
stopped.write_text(json.dumps({'session': 's', 'state': 'abandoned', 'prompt': 'p'}) + '\n')
test_fail(lambda: read_history(stopped), contains='is complete')
test_eq(len(read_history(stopped, states=None)), 1)

nested = tmp/'nested.jsonl'
nested.write_text('\n'.join(json.dumps({'session': s, 'state': 'complete', 'prompt': s})
                            for s in ('abc', 'abcd')) + '\n')
test_eq([t['prompt'] for t in read_history(nested, 'abc')], ['abc'])
test_eq([t['prompt'] for t in read_history(nested, 'abcd')], ['abcd'])

## Score a route

An assessment reads what the agent did, never what it said about it. Three faults are visible in the
archive alone: a general search when the prompt named a repository and the tool that reads one, a
failed call repeated unchanged, and an error whose own text says the call broke the tool's contract.

In [ ]:
#| export
@dataclass(frozen=True)
class Finding:
    "One tool-route problem in a recorded turn."
    kind: str
    tool: str
    index: int
    message: str

@dataclass(frozen=True)
class Assessment:
    "The route score and findings for one or more turns."
    score: int
    calls: int
    findings: tuple[Finding, ...]

    def dict(self):
        "The assessment as plain JSON-ready data."
        return {'score': self.score, 'calls': self.calls, 'findings': [asdict(f) for f in self.findings]}

def _score(findings): return max(0, 100 - 20*len(findings))

In [ ]:
#| export
RESEARCH_TOOLS = {'web_search', 'read_url', 'search_code', 'run_shell'}
PROTOCOL = (
    ('edit_cell', ('could not parse commands',),
     'Use the notebook editor command format from its current tool contract.'),
    ('run_shell', ('usage:', 'unrecognized arguments'),
     'Read the project command contract before retrying.'),
)

def _args_text(action): return ' '.join(str(v) for v in (action.get('args') or {}).values())

def _reads_repo(action):
    return action.get('tool') == 'run_shell' and 'fossick read-gh-repo' in _args_text(action)

def assess_turn(turn):
    "Assess the tool route recorded in one Ramabana turn."
    acts, findings = turn.get('activity') or [], []
    prompt = str(turn.get('prompt') or '').lower()
    if 'github' in prompt and 'fossick' in prompt:
        research = [(i, a) for i, a in enumerate(acts) if a.get('tool') in RESEARCH_TOOLS]
        if research and not _reads_repo(research[0][1]):
            i, a = research[0]
            findings.append(Finding('route', a.get('tool', ''), i,
                                    'Use fossick read-gh-repo as the first repository research call.'))
    seen = set()
    for i, a in enumerate(acts):
        if a.get('ok', True): continue
        tool, detail = a.get('tool', ''), str(a.get('detail') or '').lower()
        key = (tool, json.dumps(a.get('args') or {}, sort_keys=True, default=str))
        if key in seen:
            findings.append(Finding('repeat_failure', tool, i,
                                    'Diagnose or change route before repeating a failed call.'))
        seen.add(key)
        for name, needles, message in PROTOCOL:
            if tool == name and any(n in detail for n in needles):
                findings.append(Finding('tool_protocol', tool, i, message))
                break
    return Assessment(_score(findings), len(acts), tuple(findings))

def assess_history(turns):
    "Assess several turns as one route corpus."
    each = [assess_turn(t) for t in turns]
    findings = tuple(f for a in each for f in a.findings)
    return Assessment(_score(findings), sum(a.calls for a in each), findings)

The first research call decides whether a repository request took the intended route.

In [ ]:
bad_route = {'prompt': 'Use fossick to research https://github.com/AnswerDotAI/llmdojo',
             'activity': [{'tool': 'web_search', 'ok': True, 'args': {'query': 'llmdojo'}}]}
test_eq([f.kind for f in assess_turn(bad_route).findings], ['route'])

good_route = {'prompt': 'Use fossick to research https://github.com/AnswerDotAI/llmdojo',
              'activity': [{'tool': 'run_shell', 'ok': True,
                            'args': {'command': 'fossick read-gh-repo https://github.com/AnswerDotAI/llmdojo'}}]}
test_eq(assess_turn(good_route), Assessment(100, 1, ()))

A malformed edit sent twice records both the protocol fault and the failure to change route.

In [ ]:
edit = {'tool': 'edit_cell', 'ok': False, 'args': {'commands': ''},
        'detail': 'could not parse commands: Expecting value'}
test_eq([f.kind for f in assess_turn({'activity': [edit, dict(edit)]}).findings],
        ['tool_protocol', 'repeat_failure', 'tool_protocol'])

A call that succeeded costs nothing, and neither does one whose record predates the `ok` flag:
Ramabana's own default for a call in flight is success, so a missing flag is not a failure.

In [ ]:
test_eq(assess_turn({'activity': [{'tool': 'read_file', 'args': {}}]}), Assessment(100, 1, ()))
test_eq(assess_history([bad_route, good_route]).calls, 2)
test_eq(assess_history([]), Assessment(100, 0, ()))

## Command line

`drona` prints one JSON report, on the same archive, prefix and state rules as everything else.

In [ ]:
#| export
@call_parse
def main(
    history: str=str(RAMABANA_HISTORY), # Ramabana history path
    session: str=None,                  # session id, id prefix, or `latest`
    every_state: bool=False,            # score abandoned and failed turns too
):
    "Assess persisted Ramabana tool routes."
    turns = report(read_history, history, session, states=None if every_state else DONE)
    print(json.dumps(assess_history(turns).dict(), indent=2))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()